In [1]:
import os
import re
from dotenv import load_dotenv
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Load Environment Variables
load_dotenv("../.env")

# 2. Text Preprocessing & Cleaning Function
def clean_text(raw_text):
    cleaned = re.sub(r'\n+', '\n', raw_text)
    cleaned = re.sub(r' +', ' ', cleaned)
    return cleaned.strip()

# Load raw markdown data
file_path = "../data/manual_admission_registration.md"
with open(file_path, "r", encoding="utf-8") as f:
    raw_markdown = f.read()

cleaned_markdown = clean_text(raw_markdown)

# 3. Demonstration of Tokenization (Requirement for Checkpoint 1)
tokenizer = tiktoken.get_encoding("cl100k_base")
tokens = tokenizer.encode(cleaned_markdown)

print("--- TEXT PREPROCESSING & TOKENIZATION DEMO ---")
print(f"Raw Character Count: {len(raw_markdown)}")
print(f"Cleaned Character Count: {len(cleaned_markdown)}")
print(f"Total Token Count: {len(tokens)}")
print(f"Sample Tokens (First 10 IDs): {tokens[:10]}")
print(f"Decoded Sample: '{tokenizer.decode(tokens[:10])}'\n")

# 4. Chunking (Text Splitting)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_text(cleaned_markdown)
print(f"Total Document Chunks Generated: {len(chunks)}")
print(f"Sample Chunk 1 Preview:\n{chunks[0]}\n")

# 5. Embedding Generation Demo (Free Local HuggingFace Model)
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
sample_vector = embeddings_model.embed_query(chunks[0])

print("--- EMBEDDING GENERATION DEMO ---")
print(f"Embedding Vector Dimension: {len(sample_vector)}")
print(f"Sample Vector Values (First 5 Dimensions): {sample_vector[:5]}")

--- TEXT PREPROCESSING & TOKENIZATION DEMO ---
Raw Character Count: 5782
Cleaned Character Count: 5733
Total Token Count: 1145
Sample Tokens (First 10 IDs): [2, 63446, 323, 25532, 14881, 198, 334, 11916, 70858, 9941]
Decoded Sample: '# Admission and Registration Manual
**Academic Year'

Total Document Chunks Generated: 16
Sample Chunk 1 Preview:
# Admission and Registration Manual
**Academic Year: 2026-2027**
**Document Version:** 1.0
## 1. General Admission Policies



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

--- EMBEDDING GENERATION DEMO ---
Embedding Vector Dimension: 384
Sample Vector Values (First 5 Dimensions): [0.028309786692261696, -0.04001559689640999, 0.09243123978376389, -0.034366119652986526, -0.05422051250934601]


In [2]:
import os
from langchain_community.vectorstores import Chroma

# Fixes for Windows Jupyter deadlocks
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

print("1. Preparing ChromaDB configuration...")
persist_directory = "../chroma_db"

print("2. Generating and saving vectors (Please wait 10-20 seconds)...")
vector_db = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings_model,
    persist_directory=persist_directory
)

print(f"3. Successfully saved {len(chunks)} document chunks to ChromaDB.")

C:\Users\63916\AppData\Local\Temp\ipykernel_2272\3236342732.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


1. Preparing ChromaDB configuration...
2. Generating and saving vectors (Please wait 10-20 seconds)...
3. Successfully saved 16 document chunks to ChromaDB.
